# Buscador de Papers - PubMed / NCBI

**PubMed** es la base de datos de literatura biomedica del gobierno de EE.UU. (NCBI/NIH):
- +36 millones de articulos de **medicina, biologia, farmacologia, salud publica**
- API completamente **gratuita** (con API key obtiene mayor velocidad)
- Incluye papers de **PubMed Central (PMC)** con texto completo gratuito
- Ideal para investigacion en **ciencias de la vida y salud**

---

## Paso 1: Instalar dependencias

In [ ]:
!pip install requests pandas sumy nltk fpdf2 biopython --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('Listo.')

## Paso 2: Importar librerias y configuracion

In [ ]:
import requests
import pandas as pd
import datetime
import time
import xml.etree.ElementTree as ET
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words
from fpdf import FPDF
from google.colab import files
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── OPCIONAL: API key gratuita de NCBI para 10 req/seg (sin key = 3 req/seg) ──
# Registro: https://www.ncbi.nlm.nih.gov/account/
# IMPORTANTE: No compartas este notebook con tu API key escrita aqui
NCBI_API_KEY = None   # Ejemplo: 'abc123def456ghi789'

BASE_ESEARCH = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
BASE_EFETCH  = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'

def ncbi_params(extra=None):
    p = {'retmode': 'json'}
    if NCBI_API_KEY:
        p['api_key'] = NCBI_API_KEY
    if extra:
        p.update(extra)
    return p

print('Librerias importadas.')

## Paso 3: Funciones principales

In [ ]:
def resumir_texto(texto, oraciones=3):
    if not texto or len(texto) < 80:
        return texto or 'Sin abstract disponible.'
    try:
        parser = PlaintextParser.from_string(texto, Tokenizer('english'))
        summarizer = LsaSummarizer(Stemmer('english'))
        summarizer.stop_words = get_stop_words('english')
        return ' '.join(str(s) for s in summarizer(parser.document, oraciones))
    except Exception:
        return texto[:600] + '...' if len(texto) > 600 else texto


def obtener_ids(query, max_results, sort, fecha_desde, fecha_hasta, solo_free_full):
    params = ncbi_params({
        'db':         'pubmed',
        'term':       query,
        'retmax':     min(max_results, 200),
        'usehistory': 'y',
    })
    sort_map = {'relevance': 'relevance', 'reciente': 'pub date'}
    params['sort'] = sort_map.get(sort, 'relevance')

    if fecha_desde or fecha_hasta:
        params['mindate'] = fecha_desde or '1900'
        params['maxdate'] = fecha_hasta or '3000'
        params['datetype'] = 'pdat'

    if solo_free_full:
        params['term'] = f'({query}) AND free full text[sb]'

    resp = requests.get(BASE_ESEARCH, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()['esearchresult']
    return data.get('idlist', []), data.get('count', 0)


def obtener_detalles(ids):
    if not ids:
        return []
    params = {
        'db':      'pubmed',
        'id':      ','.join(ids),
        'retmode': 'xml',
        'rettype': 'abstract',
    }
    if NCBI_API_KEY:
        params['api_key'] = NCBI_API_KEY

    resp = requests.get(BASE_EFETCH, params=params, timeout=40)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)

    resultados = []
    for art in root.findall('.//PubmedArticle'):
        mc = art.find('.//MedlineCitation')
        a  = mc.find('Article') if mc is not None else None
        if a is None:
            continue

        titulo = (a.findtext('ArticleTitle') or 'Sin titulo').strip()

        autores_list = []
        for au in (a.findall('.//Author') or [])[:8]:
            ln = au.findtext('LastName') or ''
            fn = au.findtext('ForeName') or ''
            if ln:
                autores_list.append(f'{ln} {fn}'.strip())
        autores = ', '.join(autores_list) or 'N/A'

        abstract_parts = [t.text or '' for t in a.findall('.//AbstractText')]
        abstract = ' '.join(abstract_parts).strip()

        pub    = a.find('.//PubDate')
        anio   = pub.findtext('Year') if pub is not None else 'N/A'
        mes    = pub.findtext('Month') if pub is not None else ''
        fecha  = f'{anio} {mes}'.strip()
        revista = a.findtext('.//Title') or a.findtext('.//ISOAbbreviation') or 'N/A'

        pmid = mc.findtext('PMID') if mc is not None else 'N/A'
        doi  = 'N/A'
        pmc  = 'N/A'
        for aid in art.findall('.//ArticleId'):
            if aid.get('IdType') == 'doi':
                doi = aid.text or 'N/A'
            if aid.get('IdType') == 'pmc':
                pmc = aid.text or 'N/A'

        mesh = ', '.join(
            d.findtext('DescriptorName') or ''
            for d in (mc.findall('.//MeshHeading') if mc is not None else [])[:6]
        )

        resultados.append({
            'titulo':   titulo,
            'autores':  autores,
            'fecha':    fecha,
            'anio':     anio,
            'revista':  revista,
            'pmid':     pmid,
            'doi':      doi,
            'pmc':      pmc,
            'mesh':     mesh,
            'abstract': abstract,
            'resumen':  resumir_texto(abstract),
            'url':      f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/' if pmid != 'N/A' else '',
            'pdf_url':  f'https://www.ncbi.nlm.nih.gov/pmc/articles/{pmc}/' if pmc != 'N/A' else '',
        })
    return resultados


def buscar_papers(query, max_results=10, sort='relevance',
                  fecha_desde=None, fecha_hasta=None, solo_free_full=False):
    ids, total = obtener_ids(query, max_results, sort, fecha_desde, fecha_hasta, solo_free_full)
    if not ids:
        return []
    time.sleep(0.35)  # Respetar limite NCBI (3 req/seg sin API key)
    return obtener_detalles(ids)


def mostrar_html(resultados):
    if not resultados:
        display(HTML('<p style="color:red">No se encontraron resultados.</p>'))
        return
    html = f'<h3>Se encontraron {len(resultados)} articulos:</h3>'
    for i, p in enumerate(resultados, 1):
        pmc_link = (f'<a href="{p["pdf_url"]}" target="_blank">Texto completo PMC</a>'
                    if p['pdf_url'] else '<span style="color:#aaa">Sin texto PMC</span>')
        html += f'''
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;
                    background:#f9f9f9;font-family:Arial,sans-serif;">
          <h4 style="margin:0 0 6px;color:#1a0dab;">#{i} — {p['titulo']}</h4>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Autores:</b> {p['autores']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Fecha:</b> {p['fecha']} &nbsp;|
            <b>Revista:</b> {p['revista']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>PMID:</b> {p['pmid']} &nbsp;|
            <b>DOI:</b> {p['doi']} &nbsp;|
            <b>PMC:</b> {p['pmc']}</p>
          <p style="margin:2px 0;font-size:12px;color:#777;">
            <b>MeSH:</b> {p['mesh'] or 'N/A'}</p>
          <details style="margin-top:8px;">
            <summary style="cursor:pointer;color:#1a0dab;font-size:13px;">Ver abstract completo</summary>
            <p style="font-size:12px;color:#333;margin-top:6px;">{p['abstract'] or 'No disponible.'}</p>
          </details>
          <div style="background:#eef6ff;border-left:4px solid #1a73e8;
                      padding:10px;margin-top:10px;border-radius:4px;">
            <b style="font-size:13px;">Resumen automatico:</b>
            <p style="font-size:13px;margin:4px 0;">{p['resumen']}</p>
          </div>
          <p style="margin-top:8px;font-size:13px;">
            <a href="{p['url']}" target="_blank">Ver en PubMed</a>
            &nbsp;|&nbsp; {pmc_link}
          </p>
        </div>'''
    display(HTML(html))


def exportar_csv(resultados, nombre=None):
    if not resultados: print('Sin resultados.'); return
    nombre = nombre or f'pubmed_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    pd.DataFrame(resultados).to_csv(nombre, index=False, encoding='utf-8-sig')
    print(f'Archivo: {nombre}')
    files.download(nombre)


def exportar_pdf(resultados, nombre=None):
    if not resultados: print('Sin resultados.'); return
    nombre = nombre or f'pubmed_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.pdf'
    safe = lambda t: (str(t) or '').encode('latin-1', 'replace').decode('latin-1')
    pdf = FPDF()
    pdf.set_auto_page_break(True, 15)
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 16)
    pdf.cell(0, 10, 'Resultados - PubMed / NCBI', ln=True, align='C')
    pdf.set_font('Helvetica', '', 10)
    pdf.cell(0, 8, f'Generado: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}', ln=True, align='C')
    pdf.ln(6)
    for i, p in enumerate(resultados, 1):
        pdf.set_font('Helvetica', 'B', 12)
        pdf.multi_cell(0, 7, safe(f'{i}. {p["titulo"]}'))
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe(f'Autores: {p["autores"]}'))
        pdf.multi_cell(0, 6, safe(f'Fecha: {p["fecha"]}  |  Revista: {p["revista"]}'))
        pdf.multi_cell(0, 6, safe(f'PMID: {p["pmid"]}  |  DOI: {p["doi"]}  |  PMC: {p["pmc"]}'))
        pdf.multi_cell(0, 6, safe(f'MeSH: {p["mesh"]}'))
        pdf.ln(2)
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Resumen automatico:', ln=True)
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe(p['resumen']))
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Abstract:', ln=True)
        pdf.set_font('Helvetica', '', 8)
        pdf.multi_cell(0, 5, safe(p['abstract']))
        pdf.ln(4)
        pdf.set_draw_color(180, 180, 180)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(4)
    pdf.output(nombre)
    print(f'Archivo: {nombre}')
    files.download(nombre)


print('Funciones cargadas.')

## Paso 4: Interfaz interactiva

> **Tip busqueda avanzada PubMed:** puedes usar operadores como `AND`, `OR`, `[tiab]` (titulo/abstract), `[au]` (autor), `[jour]` (revista). Ejemplo: `COVID-19[tiab] AND vaccine[tiab] AND 2022[pdat]`

In [ ]:
w_query   = widgets.Text(value='', placeholder='Ej: diabetes machine learning, cancer immunotherapy...',
                         description='Busqueda:', layout=widgets.Layout(width='75%'),
                         style={'description_width': '80px'})
w_sort    = widgets.RadioButtons(
    options=[('Relevancia','relevance'),('Mas recientes','reciente')],
    value='relevance', description='Ordenar:', style={'description_width': '80px'})
w_max     = widgets.IntSlider(value=10, min=1, max=200, step=5, description='Cantidad:',
                               layout=widgets.Layout(width='60%'), style={'description_width': '80px'})
w_free    = widgets.Checkbox(value=False, description='Solo articulos con texto completo gratuito (PMC)', indent=False)
w_desde   = widgets.Text(value='2015', description='Desde:', placeholder='YYYY',
                          style={'description_width': '60px'}, layout=widgets.Layout(width='140px'))
w_hasta   = widgets.Text(value='2024', description='Hasta:', placeholder='YYYY',
                          style={'description_width': '60px'}, layout=widgets.Layout(width='140px'))
w_filtrar = widgets.Checkbox(value=False, description='Filtrar por rango de anios', indent=False)

btn_buscar = widgets.Button(description='Buscar', button_style='primary', icon='search',
                             layout=widgets.Layout(width='130px', height='36px'))
btn_csv    = widgets.Button(description='CSV', button_style='success', icon='download',
                             layout=widgets.Layout(width='110px', height='36px'), disabled=True)
btn_pdf    = widgets.Button(description='PDF', button_style='warning', icon='file',
                             layout=widgets.Layout(width='110px', height='36px'), disabled=True)

out_status  = widgets.Output()
out_results = widgets.Output()
resultados_globales = []

def on_buscar(b):
    global resultados_globales
    query = w_query.value.strip()
    if not query:
        with out_status: clear_output(); print('Escribe un termino de busqueda.')
        return
    btn_buscar.disabled = True; btn_csv.disabled = True; btn_pdf.disabled = True
    with out_status: clear_output(); print(f'Buscando en PubMed: "{query}"...')
    with out_results: clear_output()
    try:
        desde = w_desde.value if w_filtrar.value else None
        hasta = w_hasta.value if w_filtrar.value else None
        resultados_globales = buscar_papers(query, w_max.value, w_sort.value,
                                             desde, hasta, w_free.value)
        with out_status: clear_output(); print(f'{len(resultados_globales)} articulos encontrados.')
        with out_results: mostrar_html(resultados_globales)
        btn_csv.disabled = False; btn_pdf.disabled = False
    except Exception as e:
        with out_status: clear_output(); print(f'Error: {e}')
    finally:
        btn_buscar.disabled = False

btn_buscar.on_click(on_buscar)
btn_csv.on_click(lambda b: exportar_csv(resultados_globales))
btn_pdf.on_click(lambda b: exportar_pdf(resultados_globales))

display(HTML('<h3 style="font-family:Arial">Buscador de Papers - PubMed / NCBI</h3>'))
display(widgets.VBox([
    w_query,
    widgets.HBox([w_sort, w_max]),
    w_free,
    widgets.HBox([w_filtrar, w_desde, w_hasta]),
    widgets.HBox([btn_buscar, btn_csv, btn_pdf]),
    out_status, out_results
]))

## Paso 5 (opcional): Busqueda por codigo directo

In [ ]:
QUERY       = 'CRISPR gene editing cancer therapy'  # Busqueda PubMed
MAX         = 5                                      # Cantidad (max 200)
SORT        = 'reciente'                             # 'relevance' o 'reciente'
DESDE       = '2020'                                 # Anio inicio (None para sin filtro)
HASTA       = '2024'                                 # Anio fin (None para sin filtro)
SOLO_FREE   = False                                  # True = solo PMC free full text

resultados = buscar_papers(QUERY, MAX, SORT, DESDE, HASTA, SOLO_FREE)
mostrar_html(resultados)

# exportar_csv(resultados)
# exportar_pdf(resultados)

---

## Licencias, Politicas de Datos y Uso de la API

### Quien opera PubMed / NCBI
PubMed es operado por el **National Center for Biotechnology Information (NCBI)**, parte de los **National Institutes of Health (NIH)** del gobierno de los Estados Unidos. Es un recurso publico financiado con impuestos federales estadounidenses, por lo que su acceso es gratuito y universal.

---

### Licencia de los datos

| Tipo de dato | Licencia |
|---|---|
| Metadatos de PubMed (PMID, titulo, autores, abstract, MeSH, revista, fecha) | **Dominio publico** — obra del gobierno de EE.UU. sin restricciones de copyright |
| Textos completos en PubMed Central (PMC) | **Varia por paper** — ver tabla abajo |
| API E-utilities (herramienta de busqueda) | **Gratuita, dominio publico** |
| Terminos MeSH (Medical Subject Headings) | **Dominio publico** — gestionados por la Biblioteca Nacional de Medicina (NLM) |

Los textos completos en **PubMed Central** tienen licencias individuales:

| Licencia en PMC | Que permite |
|---|---|
| **CC BY** | Usar, copiar, distribuir y adaptar con atribucion al autor |
| **CC BY-NC** | Solo uso **no comercial** con atribucion |
| **CC BY-NC-ND** | Solo lectura, no comercial, sin modificaciones |
| **CC BY-NC-SA** | No comercial, obras derivadas con misma licencia |
| **Sin licencia CC (Publisher)** | Solo lectura en el sitio; redistribucion no permitida |
| **NIH Manuscript** | Manuscritos de investigacion financiada por NIH — uso libre en investigacion |

> Los abstracts de PubMed son siempre de dominio publico. El texto completo del paper puede tener restricciones segun la editorial.

---

### Terminos de uso de la API (NCBI E-utilities)

- **Completamente gratuita** sin registro
- **Limite sin API key:** 3 peticiones por segundo
- **Con API key gratuita** (registro en NCBI): 10 peticiones por segundo
- **Registro gratuito para API key:** https://www.ncbi.nlm.nih.gov/account/
- **Permitido:** busqueda de metadatos, descarga de abstracts, uso en investigacion academica y clinica
- **No permitido:** descargas masivas automatizadas de texto completo sin autorizacion de las editoriales
- **Obligatorio con API key:** incluir tu email en las peticiones para que NCBI pueda contactarte si hay problemas
- Politica oficial: https://www.ncbi.nlm.nih.gov/home/about/policies/
- Documentacion E-utilities: https://www.ncbi.nlm.nih.gov/books/NBK25501/

**Como obtener y usar una API key de NCBI:**
1. Crea una cuenta gratuita en https://www.ncbi.nlm.nih.gov/account/
2. Ve a tu perfil > API Key Management
3. Genera tu key y pegala en el Paso 2 de este notebook:
```python
NCBI_API_KEY = 'tu_api_key_aqui'
```

---

### Privacidad y seguridad de datos

- Este notebook **no recopila ni almacena** datos personales del usuario
- Las consultas se envian directamente desde tu sesion (Colab/Kaggle) a los servidores del NCBI
- NCBI registra el uso de su API para monitoreo y puede contactarte si detecta uso abusivo
- Si ingresas tu API key en el notebook, esta queda visible en tu sesion de Colab/Kaggle. **No compartas el notebook con tu API key escrita**; borrarla antes de compartir
- Los archivos CSV y PDF generados se descargan localmente; **no se transmiten a servidores externos**
- NCBI tiene una politica de privacidad completa en: https://www.nlm.nih.gov/privacy.html

---

### Busqueda avanzada con sintaxis PubMed

PubMed admite una sintaxis de busqueda avanzada muy potente. Algunos ejemplos:

| Sintaxis | Significado | Ejemplo |
|---|---|---|
| `[tiab]` | Buscar en titulo y abstract | `COVID-19[tiab]` |
| `[au]` | Buscar por autor | `Smith J[au]` |
| `[jour]` | Buscar en revista especifica | `Nature[jour]` |
| `[mh]` | Buscar por termino MeSH | `Diabetes Mellitus[mh]` |
| `[pdat]` | Filtrar por fecha de publicacion | `2020:2024[pdat]` |
| `AND / OR / NOT` | Operadores logicos | `CRISPR AND cancer NOT review` |
| `free full text[sb]` | Solo articulos con PDF gratuito | `vaccine AND free full text[sb]` |

---

### Nota de citacion academica

Para citar PubMed como fuente de datos en publicaciones:
> National Library of Medicine (US). PubMed [Internet]. Bethesda (MD): National Center for Biotechnology Information (NCBI); 1996 – [cited YYYY Mon DD]. Available from: https://pubmed.ncbi.nlm.nih.gov/

Cada paper debe citarse individualmente usando su PMID o DOI.